In [53]:
import pandas as pd
from scipy.stats import poisson
from scipy.stats import chisquare
import numpy as np
from scipy.stats import norm

df = pd.read_csv("../data/crashes_clean.csv", low_memory = False, parse_dates=['crash_dt'])
df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5,crash_dt,year,month,weekday,hour,injury_crash,fatal_crash,borough_filled
0,12/31/2025,21:23,BROOKLYN,11236.0,40.643660,-73.900665,"(40.64366, -73.900665)",ROCKAWAY PKWY,NaN,FLATLANDS AVE,...,NaN,NaN,2025-12-31 21:23:00,2025,12,Wednesday,21,True,False,BROOKLYN
1,12/31/2025,12:15,BRONX,10452.0,40.837140,-73.920480,"(40.83714, -73.92048)",E 168 ST,NaN,RIVER AVE,...,NaN,NaN,2025-12-31 12:15:00,2025,12,Wednesday,12,False,False,BRONX
2,12/31/2025,13:30,NaN,NaN,40.665054,-73.735990,"(40.665054, -73.73599)",NaN,241-42 S CONDUIT AVE,NaN,...,NaN,NaN,2025-12-31 13:30:00,2025,12,Wednesday,13,False,False,QUEENS
3,12/31/2025,05:20,BROOKLYN,11207.0,40.657814,-73.896126,"(40.657814, -73.896126)",LINDEN BLVD,NaN,WILLIAMS AVE,...,NaN,NaN,2025-12-31 05:20:00,2025,12,Wednesday,5,False,False,BROOKLYN
4,12/31/2025,22:03,BRONX,10468.0,40.859978,-73.900790,"(40.859978, -73.90079)",E 184 ST,NaN,MORRIS AVE,...,NaN,NaN,2025-12-31 22:03:00,2025,12,Wednesday,22,False,False,BRONX


In [54]:
#making a date column
df['date'] = df['crash_dt'].dt.date
df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,VEHICLE TYPE CODE 5,crash_dt,year,month,weekday,hour,injury_crash,fatal_crash,borough_filled,date
0,12/31/2025,21:23,BROOKLYN,11236.0,40.643660,-73.900665,"(40.64366, -73.900665)",ROCKAWAY PKWY,NaN,FLATLANDS AVE,...,NaN,2025-12-31 21:23:00,2025,12,Wednesday,21,True,False,BROOKLYN,2025-12-31
1,12/31/2025,12:15,BRONX,10452.0,40.837140,-73.920480,"(40.83714, -73.92048)",E 168 ST,NaN,RIVER AVE,...,NaN,2025-12-31 12:15:00,2025,12,Wednesday,12,False,False,BRONX,2025-12-31
2,12/31/2025,13:30,NaN,NaN,40.665054,-73.735990,"(40.665054, -73.73599)",NaN,241-42 S CONDUIT AVE,NaN,...,NaN,2025-12-31 13:30:00,2025,12,Wednesday,13,False,False,QUEENS,2025-12-31
3,12/31/2025,05:20,BROOKLYN,11207.0,40.657814,-73.896126,"(40.657814, -73.896126)",LINDEN BLVD,NaN,WILLIAMS AVE,...,NaN,2025-12-31 05:20:00,2025,12,Wednesday,5,False,False,BROOKLYN,2025-12-31
4,12/31/2025,22:03,BRONX,10468.0,40.859978,-73.900790,"(40.859978, -73.90079)",E 184 ST,NaN,MORRIS AVE,...,NaN,2025-12-31 22:03:00,2025,12,Wednesday,22,False,False,BRONX,2025-12-31


In [55]:
daily_fatal = df.groupby('date')['fatal_crash'].sum()
len(daily_fatal)

1096

In [56]:
# is the daily mean and variance the same? Follow Poisson Distribution?
daily_fatal.agg(['mean', 'var'])

mean    0.672445
var     0.684391
Name: fatal_crash, dtype: float64

The mean (0.672) and variance (0.684) are very close, which is consistent with a Poisson distribution. Next, check whether the full shape matches.

In [57]:
#observed counts
observed = daily_fatal.value_counts().sort_index() 
observed

fatal_crash
0    575
1    339
2    152
3     26
4      4
Name: count, dtype: int64

In [58]:
#predicted counts
lamda = daily_fatal.mean()
predicted = poisson.pmf(observed.index, lamda) * len(daily_fatal)
pd.DataFrame({'observed' : observed, 'predicted' : predicted})

,observed,predicted
fatal_crash,,
0,575,559.462898
1,339,376.208171
2,152,126.489700
3,26,28.352466
4,4,4.766370


Compared with Poisson, there are more days with 0 deaths (575 vs 559.5 predicted) and
2 deaths (152 vs 126.5), and fewer days with 1 death (339 vs 376.2). Days with 3–4 deaths
roughly match. The real data is more spread out than Poisson expects, which could mean
risk isn't the same every day. A chi-square goodness-of-fit test will check whether
these gaps are larger than chance.

In [59]:
#Chi-square to compare the data distribution to the theoretical poisson distribution
#grouping group 3 and 4 since 4 has a small predicted count
#χ² = Σ (actual − predicted)² / predicted
observed_list = [(daily_fatal == 0).sum(), (daily_fatal == 1).sum(), (daily_fatal == 2).sum(), (daily_fatal >= 3).sum()]
expected_list = [poisson.pmf(0, lamda) * len(daily_fatal),
            poisson.pmf(1, lamda) * len(daily_fatal),
            poisson.pmf(2, lamda) * len(daily_fatal),
            poisson.sf(2, lamda) * len(daily_fatal)]
chisquare(observed_list, expected_list, ddof = 1)

Power_divergenceResult(statistic=np.float64(9.691961921491062), pvalue=np.float64(0.007859903415305985))

Since the p-value is 0.008 < 0.05, we reject the null hypothesis which states that the distribution of the data matches the poisson distribution.
The mean and variance were close but the chi square goodness of fit test rejects, suggests that the risk for fatal crash per day may not  be constant day to day. The chi square tests the whole distrubution instead of just two numbers, mean and variance. The poisson distribution underestimates how often quiet days and bad days happen.

------------------------------------------------------------------------------------------------------------------------------------------
When someone gets hurt in a crash, does it matter who was hurt? In other words, is a crash more likely to be fatal when the victim is a pedestrian, a cyclist or a motorist?

In probability terms, compare:
- P(fatal | a pedestrian was injured or killed)
- P(fatal | a cyclist was injured or killed)
- P(fatal | a motorist was injured or killed)

In [60]:
df['ped_hurt'] = (df['NUMBER OF PEDESTRIANS INJURED'] + df['NUMBER OF PEDESTRIANS KILLED']) > 0
df['cyc_hurt'] = (df['NUMBER OF CYCLIST INJURED'] + df['NUMBER OF CYCLIST KILLED']) > 0
df['motor_hurt'] = (df['NUMBER OF MOTORIST INJURED'] + df['NUMBER OF MOTORIST KILLED']) > 0


victim_table = pd.DataFrame(
    {'victims' : ['pedestrian', 'cyclist', 'motorist'],
     'crashes': [df['ped_hurt'].sum(), df['cyc_hurt'].sum(), df['motor_hurt'].sum()],
     'fatal' : [df[df['ped_hurt']]['fatal_crash'].sum(), df[df['cyc_hurt']]['fatal_crash'].sum(), df[df['motor_hurt']]['fatal_crash'].sum()],
     'p_fatal' : [df[df['ped_hurt']]['fatal_crash'].mean(), df[df['cyc_hurt']]['fatal_crash'].mean(), df[df['motor_hurt']]['fatal_crash'].mean()]
})
victim_table

,victims,crashes,fatal,p_fatal
0,pedestrian,26986,346,0.012821
1,cyclist,14723,63,0.004279
2,motorist,70626,319,0.004517


When a pedestrian is hurt, the crash is fatal 1.28% of the time, about 3 times the rate for cyclists (0.43%) or motorists (0.45%). Cyclists and motorists are almost the same. We can do a two proportion z test to determine if this gap is significant.

In [61]:
# Compare pedestrian hurt against no pedestrian hurt, but only among injury crashes.
ped_table = df[df['injury_crash']].groupby('ped_hurt')['fatal_crash'].agg(['sum', 'size', 'mean'])
ped_table


,sum,size,mean
ped_hurt,,,
False,391,91142,0.004290
True,346,26986,0.012821


In [62]:
p_hurt = ped_table.loc[True, 'mean']
p_nothurt = ped_table.loc[False, 'mean']
p_pooled = ped_table['sum'].sum() / ped_table['size'].sum()
se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / ped_table.loc[True, 'size'] + 1 / ped_table.loc[False, 'size']))
z = (p_hurt - p_nothurt) / se
print(z)

p_value = 2 * norm.sf(z)
p_value

15.634243380427208


np.float64(4.2549882087828196e-55)

Among injury crashes, the crash was fatal 1.28% of the time when a pedestrian was hurt,
vs 0.43% when no pedestrian was hurt, about 3 times higher (z = 15.6, p < 0.001).
We reject H₀: the difference is very unlikely to be due to chance.

- **Why it might happen:** pedestrians have no vehicle protecting them, but this data
  can't confirm the cause.
- **Groups:** `ped_hurt` puts every injury crash in exactly one group, so the two samples
  don't overlap. (Pedestrian vs motorist would overlap: 543 crashes had both hurt.)
- **Limitation:** the test assumes crashes are independent. Some may share conditions such
  as weather or day, so the p-value may be somewhat optimistic, but with z = 15.6 the
  conclusion is unlikely to change.
- **Data note:** pedestrians are only recorded if hurt, so this compares crashes where a
  pedestrian was *hurt*, not every crash involving a pedestrian.